In [5]:
!pip install librosa soundfile

import librosa
import soundfile as sf
import numpy as np
import IPython.display as ipd

def reduce_noise_spectral(audio_path, output_path):
    y, sr = librosa.load(audio_path, sr=None)

    stft = librosa.stft(y)
    stft_mag, stft_phase = librosa.magphase(stft)

    # Estimate noise profile from initial frame
    noise_power = np.mean(stft_mag[:, :12], axis=1, keepdims=True)

    # Apply spectral subtraction
    subtracted_mag = np.maximum(stft_mag - (1.8 * noise_power), 0)

    # Reconstruct audio signal
    cleaned_stft = subtracted_mag * stft_phase
    cleaned_audio = librosa.istft(cleaned_stft)

    cleaned_audio = librosa.util.normalize(cleaned_audio)
    sf.write(output_path, cleaned_audio, sr)
    return output_path

# Run pipeline
out_file = reduce_noise_spectral('task5_1.wav', 'cleaned_audio.wav')
ipd.Audio(out_file)

In [6]:
import os
import glob
import cv2
import numpy as np
import zipfile

def process_ball_detection(images_dir, output_dir="labels"):
    os.makedirs(output_dir, exist_ok=True)

    # Color Thresholds (HSV)
    blue_lower = np.array([90, 80, 50])
    blue_upper = np.array([135, 255, 255])

    red_lower1 = np.array([0, 100, 50])
    red_upper1 = np.array([10, 255, 255])
    red_lower2 = np.array([160, 100, 50])
    red_upper2 = np.array([180, 255, 255])

    kernel = np.ones((5, 5), np.uint8)
    image_paths = glob.glob(os.path.join(images_dir, "**", "*.jpg"), recursive=True)

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w, _ = img.shape
        blurred = cv2.GaussianBlur(img, (11, 11), 0)
        hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)

        # Generate Masks
        mask_blue = cv2.inRange(hsv, blue_lower, blue_upper)
        mask_red1 = cv2.inRange(hsv, red_lower1, red_upper1)
        mask_red2 = cv2.inRange(hsv, red_lower2, red_upper2)
        mask_red = cv2.bitwise_or(mask_red1, mask_red2)

        # Noise Reduction
        mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel)
        mask_red = cv2.morphologyEx(mask_red, cv2.MORPH_OPEN, kernel)

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(output_dir, f"{base_name}.txt")

        with open(txt_path, "w") as f:
            for class_id, mask in [(0, mask_blue), (1, mask_red)]:
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    if cv2.contourArea(cnt) < 150:
                        continue
                    x, y, bw, bh = cv2.boundingRect(cnt)

                    x_center = round((x + bw / 2.0) / w, 4)
                    y_center = round((y + bh / 2.0) / h, 4)
                    norm_w = round(bw / w, 4)
                    norm_h = round(bh / h, 4)

                    f.write(f"{class_id} {x_center} {y_center} {norm_w} {norm_h}\n")

    # Zip output labels
    zip_path = "labels.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                zipf.write(os.path.join(root, file), file)

    return zip_path

# Execute pipeline
!unrar x -o+ balls.rar images/
process_ball_detection("images")


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from balls.rar

Extracting  images/balls/ball_1.jpg                                        4%  OK 
Extracting  images/balls/ball_10.jpg                                      15%  OK 
Extracting  images/balls/ball_11.jpg                                      29%  OK 
Extracting  images/balls/ball_12.jpg                                      29%  OK 
Extracting  images/balls/ball_13.jpg                                      41%  OK 
Extracting  images/balls/ball_14.jpg                                      41%  OK 
Extracting  images/balls/ball_15.jpg                                      42%  OK 
Extracting  images/balls/ball_16.jpg                                      43%  OK 
Extracting  images/balls/ball_17.jpg                                      43%  OK 
Extracting  images/balls/ball_18.jpg                                 

'labels.zip'